In [109]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim

In [111]:
df=pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [113]:
df.shape

(50000, 2)

In [115]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [117]:
df.drop_duplicates(inplace=True)

In [119]:
df.shape

(49582, 2)

In [121]:
# convert to lowercase
df["review"]=df["review"].str.lower()

In [123]:
# remove url
import re
def remove_urls(text):
    text=re.sub(r"http\S+","",text) # (pattern,replace,string)
    return text
df["review"]=df["review"].apply(remove_urls)

In [125]:
#remoe punctuation
def remove_punctuation(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuation)

In [127]:
# remove html
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

In [129]:
# remove stopword
import nltk # natural language toolkit
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [131]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [133]:
def remove_stopwords(text):
    token=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in token:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)

In [135]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [139]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

In [143]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

In [147]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [151]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [153]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [158]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
Y_train_tensor=torch.tensor(Y_train.values,dtype=torch.float32).view(-1,1)
X_test_tensor=torch.tensor(X_test,dtype=torch.float32)
Y_test_tensor=torch.tensor(Y_test.values,dtype=torch.float32).view(-1,1)

In [161]:
train_set=TensorDataset(X_train_tensor,Y_train_tensor)
test_set=TensorDataset(X_test_tensor,Y_test_tensor)

In [166]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

In [202]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        # RNN layer
        self.RNN=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        # fully connected layer (output)
        self.fc=nn.Linear(hidden_size,1)
    def forward(self,x):
        out,_=self.RNN(x)
        out=self.fc(out[:,-1,:]) # batch,sequence length,hidden size
        return out

In [204]:
input_size=X_train.shape[1]
model=RNN(input_size)
criteria=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [210]:
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1) # remove 1 feature
        outputs=model(xb) # batch size,1
        outputs=torch.sigmoid(outputs.squeeze()) # add 1 feature
        loss=criteria(outputs,yb.squeeze()) # compute loss
        loss.backward()
        optimizer.step()

In [224]:
model.eval()
with torch.no_grad():
    correct_value=0
    total_value=0
    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        predicted=(torch.sigmoid(outputs)>0.5).float()
        total_value+=yb.size(0)
        correct_value+=(predicted==yb).sum().item()
    print(f"accuracy={correct_value/total_value*100}")

accuracy=85.6609861853383
